# Baseline models on DeepCrack (DRAEM / RD4AD / EfficientAD)

**RUN THIS WITH `Save Version -> Save & Run All (Commit)`.**
It then executes on Kaggle's servers with no browser attached, so a
disconnect cannot kill it, and `/kaggle/working` is preserved as the
version's output.

Settings: Accelerator **GPU T4 x2**, Internet **On**.

Each model is scored the moment it finishes, so a later failure never
costs an earlier model's numbers. Two GPUs run in parallel:
GPU 0 = DRAEM (300 epochs, batch 8), GPU 1 = RD4AD then EfficientAD.

In [ ]:
# 1 - dependencies. torch is pinned out of the resolver so anomalib cannot
# swap in a CPU-only wheel; matplotlib<3.10 avoids anomalib's tostring_rgb crash.
import subprocess, sys
from pathlib import Path

pin = subprocess.run([sys.executable, "-c",
    "import torch;print(f'torch=={torch.__version__.split(\"+\")[0]}')"],
    capture_output=True, text=True).stdout.strip()
Path("/kaggle/working/constraints.txt").write_text(pin + "\n")
print("pinning", pin)

!pip install -q -c /kaggle/working/constraints.txt "anomalib==1.1.1" lightning imgaug \
    FrEIA einops timm kornia open_clip_torch scikit-image scikit-learn \
    opencv-python-headless "matplotlib<3.10"

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpus", torch.cuda.device_count())

In [ ]:
# 2 - numpy>=2 shim for imgaug, installed as sitecustomize so SUBPROCESSES
# get it too (a notebook-only monkeypatch would not reach `python run_...py`).
shim = Path("/kaggle/working/pyshim"); shim.mkdir(exist_ok=True)
(shim / "sitecustomize.py").write_text('''
import numpy as np
if not hasattr(np, "sctypes"):
    np.sctypes = {"float": [np.float16, np.float32, np.float64],
                  "int": [np.int8, np.int16, np.int32, np.int64],
                  "uint": [np.uint8, np.uint16, np.uint32, np.uint64],
                  "complex": [np.complex64, np.complex128],
                  "others": [bool, object, bytes, str, np.void]}
for _n, _t in (("bool", bool), ("float", float), ("int", int),
               ("object", object), ("str", str)):
    if not hasattr(np, _n):
        setattr(np, _n, _t)
''')
print("shim written")

In [ ]:
# 3 - repo + data (idempotent)
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working"); TMP = Path("/kaggle/temp"); TMP.mkdir(exist_ok=True)
REPO = WORK / "Unsupervised-Crack-Detection_CVPR"
DATA = TMP / "data/deepcrack"; RAW = TMP / "deepcrack_raw"
RESULTS = WORK / "results"; RESULTS.mkdir(exist_ok=True)

def sh(cmd, **kw):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, **kw)

if not (REPO / ".git").is_dir():
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"git clone --depth 1 https://github.com/Alirezanltv/Unsupervised-Crack-Detection_CVPR.git {REPO}")
os.chdir(REPO)

if not (RAW / "train_img").is_dir():
    sh(f"wget -q https://raw.githubusercontent.com/yhlleo/DeepCrack/master/dataset/DeepCrack.zip -O {TMP}/dc.zip")
    sh(f"unzip -q -o {TMP}/dc.zip -d {RAW}")
sh(f"python p0_reproduce/arrange_from_splits.py --raw-root {RAW} "
   f"--splits p0_reproduce/splits --name deepcrack --dst {DATA} "
   f"--masks-dir {RAW}/test_lab")

In [ ]:
# 4 - launch both GPUs. Each model: train -> dump -> eval -> sweep, chained,
# so its JSONs exist the instant it finishes.
import subprocess, os

def job(model, gpu, extra=""):
    out = f"/kaggle/working/runs/{model}/s0"
    cmd = (
        f"python -u p1_sota_baselines/run_baselines.py --data {DATA} "
        f"--out /kaggle/working/runs --models {model} --seeds 0 {extra} && "
        f"python -u common/eval_maps.py --maps {out}/maps --masks {DATA}/test/masks "
        f"--calib {out}/calib --out {RESULTS}/{model}_s0_result.json && "
        f"python -u common/sweep_threshold.py --maps {out}/maps --masks {DATA}/test/masks "
        f"--calib {out}/calib --out {RESULTS}/{model}_s0_sweep.json")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
               PYTHONPATH="/kaggle/working/pyshim",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(f"/kaggle/working/{model}.log", "w")
    return subprocess.Popen(cmd, shell=True, env=env, stdout=log,
                            stderr=subprocess.STDOUT)

# GPU 0: DRAEM (97M params -> batch 8 fits a T4; 300 epochs is our stated budget)
p_draem = job("draem", 0, "--batch 8 --max-epochs 300")
# GPU 1: RD4AD now; EfficientAD after it (sequential on the same GPU)
p_rd = job("rd4ad", 1, "--batch 16 --max-epochs 200")

import time
def tail(path):
    try:
        lines = open(path, errors="replace").read().splitlines()
        return lines[-1][-120:] if lines else ""
    except FileNotFoundError:
        return "(no log yet)"

procs, eff_started = {"draem": p_draem, "rd4ad": p_rd}, False
while procs:
    time.sleep(60)
    for name in list(procs):
        rc = procs[name].poll()
        state = "running" if rc is None else f"EXIT {rc}"
        print(f"[{time.strftime('%H:%M:%S')}] {name:12s} {state:9s} | "
              f"{tail(f'/kaggle/working/{name}.log')}", flush=True)
        if rc is not None:
            del procs[name]
            if name == "rd4ad" and not eff_started:
                procs["efficientad"] = job("efficientad", 1, "--batch 1 --max-epochs 100")
                eff_started = True
                print(f"[{time.strftime('%H:%M:%S')}] efficientad launched on GPU 1", flush=True)
print("all jobs done", flush=True)

In [ ]:
# 5 - manifest + log tails for anything that failed
import json
from pathlib import Path

js = sorted(Path("/kaggle/working/results").glob("*.json"))
print(f"{len(js)} result files\n")
for f in js:
    if f.name.endswith("result.json"):
        d = json.loads(f.read_text())
        print(f"{f.stem:28s} AUROC {d['pixel_auroc']:.4f}  AP {d['pixel_ap']:.4f}")

for m in ("draem", "rd4ad", "efficientad"):
    if not (Path("/kaggle/working/results") / f"{m}_s0_result.json").exists():
        print(f"\n===== {m} did not produce results; last 15 log lines =====")
        p = Path(f"/kaggle/working/{m}.log")
        if p.exists():
            print("".join(p.read_text().splitlines(keepends=True)[-15:]))

In [ ]:
# 6 - keep the outputs small so the committed version stays light:
# JSONs + logs stay, raw maps are zipped, model checkpoints are dropped.
!cd /kaggle/working && zip -qr baseline_maps.zip runs -x "*.ckpt" "*.pt" || true
!rm -rf /kaggle/working/runs /kaggle/working/Unsupervised-Crack-Detection_CVPR/.git
!ls -la /kaggle/working /kaggle/working/results